# 1단계 모형: 관중 결정요인 회귀분석

- 데이터: `data/processed/features.csv` (만드는 순서는 `CLAUDE.md`의 '실행')
- 모형: log(관중 수) ~ 일정·날씨(모형 1) + 경기력(모형 2) + 상대팀·특별 경기(모형 3), 구단·시즌 고정효과, 구단별 군집 표준오차

**첫 칸은 데이터 검사입니다.** 모형에 쓰는 변수에 결측이 하나라도 있으면 여기서 멈춥니다.

In [ ]:
from pathlib import Path
import pandas as pd

# notebooks/ 안에서 열어도, 프로젝트 최상위에서 열어도 같은 파일을 읽는다
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data" / "processed" / "features.csv", encoding="utf-8-sig", parse_dates=["date"])

# 모형 1~3과 민감도 분석에 쓰는 변수 (CLAUDE.md '모형' 기준)
MODEL_COLS = [
    "ln_attendance", "home_team", "season",                                   # 종속변수, 고정효과
    "day_type", "night", "rain", "hot", "cold",                               # 모형 1: 일정·날씨
    "home_rank_pre", "away_rank_pre", "home_form5", "away_form5",
    "home_first_game", "away_first_game",                                     # 모형 2: 경기력
    "ln_away_fans_pre", "away_fans_missing", "derby",
    "home_opener", "last_home", "final_round",                                # 모형 3: 상대팀·특별 경기
    "sellout_proxy", "rain_alt", "hot_alt", "cold_alt",                       # 민감도 분석
]
model_df = df[df["excluded"] == 0]

# 검사: 결측이 0이 아니면 멈춘다
na = model_df[MODEL_COLS].isna().sum(); assert na.sum() == 0, f"모형 변수에 결측이 있습니다 (CLAUDE.md '실행' 순서대로 스크립트를 다시 돌리세요): {na[na > 0].to_dict()}"

print(f"검사 통과: {len(model_df)}경기 (제외 {df['excluded'].sum()}경기), 모형 변수 {len(MODEL_COLS)}개 결측 0")